In [ ]:
# Cell 1: Parse the PDF using Databricks ai_parse_document
# Default mode is NO image output write (no extra privileges needed).
# Optional mode writes images to dbfs:/tmp if you want to test image artifacts.

from pyspark.sql.functions import expr

# PDF_PATH = "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 1603-R2 - R0 EROSION AND WATER INGESTION RECOMMENDATIONS.pdf"
# PDF_PATH = "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 1502-2R1 - 7F AND 9F AFT END COMPRESSOR RUBS.pdf"
# PDF_PATH = "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 2284 - F-CLASS BALANCE WEIGHT GROOVE ENTRY SLOT RECOMMENDATIONS.pdf"
# PDF_PATH = "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 1945-R2 - F-CLASS TURBINE WHEEL INSPECTION AND MAINTENANCE RECOMMENDATIONS.pdf"
PDF_PATH = "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/TILS_new/TIL 1937-R2 - F-CLASS TURBINE WHEEL INSPECTION AND MAINTENANCE RECOMMENDATIONS.pdf"
# Keep False for easiest experimentation without write permissions.
USE_IMAGE_OUTPUT = False
IMAGE_OUTPUT_PATH = "dbfs:/tmp/madhurima/til-analysis/parsed_images/"

docs_df = spark.read.format('binaryFile').load(PDF_PATH)

if USE_IMAGE_OUTPUT:
    parse_expr = expr(
        f"ai_parse_document(content, map('version', '2.0', 'imageOutputPath', '{IMAGE_OUTPUT_PATH}'))"
    )
    print(f"Mode: image output enabled -> {IMAGE_OUTPUT_PATH}")
else:
    parse_expr = expr("ai_parse_document(content, map('version', '2.0'))")
    print("Mode: no image output path (no volume write required)")

parsed_df = docs_df.withColumn('parsed_content', parse_expr)
parsed_df = parsed_df.drop('content')

print('Parse complete. Schema:')
parsed_df.printSchema()


ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Cell 2: Inspect parsed_content.metadata
# parsed_content is VARIANT, so convert it to JSON string and inspect via Python dict.

import json
from pyspark.sql.functions import expr

print('--- parsed_content.metadata ---')

parsed_json = parsed_df.select(expr("to_json(parsed_content) as parsed_json")).first()["parsed_json"]
parsed_obj = json.loads(parsed_json) if parsed_json else {}

metadata = parsed_obj.get("metadata", {})
print(json.dumps(metadata, indent=2))


In [ ]:
# Cell 3: Reconstruct page-wise text from document.elements
# For this schema:
# - text is in element["content"]
# - page is in element["bbox"][...]["page_id"] (0-based)

from collections import defaultdict

# Rebuild parsed_obj if Cell 2 wasn't run
if "parsed_obj" not in globals():
    import json
    from pyspark.sql.functions import expr
    parsed_json = parsed_df.select(expr("to_json(parsed_content) as parsed_json")).first()["parsed_json"]
    parsed_obj = json.loads(parsed_json) if parsed_json else {}

pages = parsed_obj.get("document", {}).get("pages", [])
elements = parsed_obj.get("document", {}).get("elements", [])

# Group text-like element content by page_id
page_text = defaultdict(list)

for el in elements:
    content = (el.get("content") or "").strip()
    if not content:
        continue

    # Get first page_id from bbox
    page_id = None
    bboxes = el.get("bbox") or []
    if bboxes and isinstance(bboxes, list):
        page_id = bboxes[0].get("page_id")

    if page_id is None:
        continue

    page_text[page_id].append(content)

print(f"Total pages: {len(pages)}")
print()

for page in pages:
    page_id = page.get("id")
    page_num = (page_id + 1) if isinstance(page_id, int) else page_id  # 1-based display
    text = "\n".join(page_text.get(page_id, []))
    print(f"=== Page {page_num} ===")
    print(text or "(no text)")
    print()

In [ ]:
# Cell 4: Inspect parsed_content.document.elements
# Uses Python dict traversal from parsed_obj built in Cell 2.

from collections import Counter

# Rebuild parsed_obj if Cell 2 wasn't run.
if "parsed_obj" not in globals():
    import json
    from pyspark.sql.functions import expr
    parsed_json = parsed_df.select(expr("to_json(parsed_content) as parsed_json")).first()["parsed_json"]
    parsed_obj = json.loads(parsed_json) if parsed_json else {}

elements = parsed_obj.get("document", {}).get("elements", [])
type_counts = Counter((el.get("type") or "UNKNOWN") for el in elements)

print('--- Element type counts ---')
for t in sorted(type_counts):
    print(f"{t}: {type_counts[t]}")


In [ ]:
# Cell 5: Inspect TABLE elements only
# Compare against DS extracted tables in extracted_document.md.


# Rebuild parsed_obj if Cell 2 wasn't run.
if "parsed_obj" not in globals():
    import json
    from pyspark.sql.functions import expr
    parsed_json = parsed_df.select(expr("to_json(parsed_content) as parsed_json")).first()["parsed_json"]
    parsed_obj = json.loads(parsed_json) if parsed_json else {}

elements = parsed_obj.get("document", {}).get("elements", [])
table_elements = [el for el in elements if (el.get("type") or "").upper() == "TABLE"]
table_elements = sorted(table_elements, key=lambda el: el.get("pageNumber", 0))

print(f"Total TABLE elements: {len(table_elements)}")
print()

for el in table_elements:
    page_num = el.get("pageNumber")
    text = el.get("text", "")
    print(f"--- Table on page {page_num} ---")
    print(text or "(empty)")
    print()


In [ ]:
# Cell 6: Inspect IMAGE elements only
# Compare against DS image descriptions to assess signal vs noise.


# Rebuild parsed_obj if Cell 2 wasn't run.
if "parsed_obj" not in globals():
    import json
    from pyspark.sql.functions import expr
    parsed_json = parsed_df.select(expr("to_json(parsed_content) as parsed_json")).first()["parsed_json"]
    parsed_obj = json.loads(parsed_json) if parsed_json else {}

elements = parsed_obj.get("document", {}).get("elements", [])
image_elements = [el for el in elements if (el.get("type") or "").upper() == "IMAGE"]
image_elements = sorted(image_elements, key=lambda el: el.get("pageNumber", 0))

print(f"Total IMAGE elements: {len(image_elements)}")
print()

for el in image_elements:
    page_num = el.get("pageNumber")
    text = el.get("text", "")
    print(f"--- Image on page {page_num} ---")
    print(text or "(no description)")
    print()
